# Stage 0 — The corpus of record

The frozen two-tier ADS retrieval: 15,650 core papers plus every paper they cite. The corpus file is **local-only** (ADS terms do not permit redistributing abstracts); the public repository ships the derived graph layer under `data/graph/` (13,800 nodes, 380,361 arcs) and the bibcode manifests under `data/raw/`. Refetching requires `ADS_API_TOKEN` and yields a *new* corpus, because the ADS index is live.

In [ ]:
import os, pathlib, sys
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "scripts").is_dir() and (root / "data").is_dir(): break
    root = root.parent
else:
    raise SystemExit("repository root (containing scripts/ and data/) not found within 6 levels")
os.chdir(root); print("working directory:", os.getcwd())
assert "igraph" in {m.split("==")[0] for m in os.popen(f"{sys.executable} -m pip list --format=freeze 2>/dev/null").read().split()}, \
    f"kernel {sys.executable} lacks python-igraph: select the grb-venv kernel (see notebooks/README.md)"

In [ ]:
CORPUS = "data/raw/ads_corpus_v2_core_frozen.jsonl"  # local-only frozen corpus (ADS terms); see README
import pathlib
HAVE_CORPUS = pathlib.Path(CORPUS).exists()
print("frozen corpus present:", HAVE_CORPUS)

Verify the frozen input bundle (corpus hash, canonical-consensus hash, control-set ledger). Fails loudly on any mismatch; skipped when the corpus is absent.

In [ ]:
if HAVE_CORPUS:
    %run scripts/campaign_preflight.py
else:
    print("skipped: preflight needs the local corpus")

Optional refetch into a NEW corpus path (never overwrite the frozen link). The cited tier used `--min-cites 1`.

In [ ]:
RUN_FETCH = False  # needs ADS_API_TOKEN; writes a new corpus
NEW = "data/raw/ads_corpus_refetch.jsonl"
if RUN_FETCH:
    %run scripts/fetch_corpus.py --stage core --years 1970-2026 --out {NEW}
    %run scripts/fetch_corpus.py --stage cited --min-cites 1 --out {NEW}
    %run scripts/fetch_corpus.py --stage merge --out {NEW}
else:
    print("skipped: the frozen corpus is the input of record")

Corpus and query metadata as a named product.

In [ ]:
if HAVE_CORPUS:
    %run scripts/corpus_meta.py
else:
    print("skipped; product of record: data/communities/corpus_meta.json")